# Lab 02C: Routing — Four Strategies, One Task

**Week 2 — Agentic AI: Building Autonomous Intelligent Systems**

A **router** looks at an incoming request and decides *where it should go* — which handler, which chain, which specialist. It is the branching point of every conditional pipeline. In this lab you build the **same customer-support router four different ways** and put them head-to-head on a shared test set, so you can feel the trade-offs yourself.

## Introduction

This lesson answers:

- What is a router, and where does it sit in an agent pipeline?
- What are the four routing strategies (rule-based, classifier, embedding, LLM), and when is each the right choice?
- How do you add a **confidence gate** so an unsure router escalates instead of guessing?
- How do you **measure** which router is best on your own data?

## Learning Goals

After completing this lesson you will be able to:

- Define routing and explain the four strategies.
- Implement three routers — **rule-based**, **classifier**, and **LLM-as-router**.
- Add a confidence gate that escalates ambiguous requests to a human.
- Compare routers head-to-head and read an accuracy table.
- Recognize the trade-offs (cost, speed, transparency) that decide which router to use.

(The fourth strategy, **embedding / semantic** routing, is previewed here and built in Week 8 alongside RAG.)

## What is routing?

Routing is a single decision: **read the request, pick the destination.** The destinations stay fixed; only the *engine* that makes the decision changes. That is the whole lab — one task, four engines.

```
                                       ┌──▶ [ billing ]
                                       │
                                       ├──▶ [ technical ]
  ┌──────────────────┐   ┌────────┐    │
  │ Incoming message │──▶│ ROUTER │────┤
  └──────────────────┘   └────────┘    │
                             ▲         ├──▶ [ account ]
                             │         │
                             │         └──▶ [ general ]
        engine = rules ──────┘
        · classifier · embeddings · LLM      (swap the engine;
                                              the rest stays the same)
```

| # | Strategy | How it decides | Best when |
| :--- | :--- | :--- | :--- |
| 1 | **Rule-Based** | explicit keywords / metadata / fixed conditions | workflows are predictable and need tight control |
| 2 | **Classifier-Based** | an LLM assigns the request to a predefined category | the possible destinations are known in advance |
| 3 | **Embedding / Semantic** | similarity between the request and route examples | users phrase the same intent many different ways |
| 4 | **LLM-as-Router** | an LLM *reasons* about the request and picks a path | tasks are ambiguous or context-dependent |

> This lab is on the **API-key** track (the full `google-genai` SDK) because the classifier and LLM routers call the Gemini API (and the Week 8 embedding router will use the embeddings endpoint).

## Use cases

Routing shows up wherever one entry point must serve many kinds of request:

- **Customer support** — send billing, technical, and account questions to different queues or playbooks (this lab).
- **Conditional pipelines** — pick which prompt chain to run for a given input (see `lab02_E`).
- **Tool / skill selection** — choose which tool or sub-agent should handle a step.
- **Triage and escalation** — answer simple requests directly, escalate ambiguous or high-risk ones to a human.
- **Cost control** — route easy requests to a cheap path and spend a bigger model only on the hard ones.

## Building blocks

To build a router you need:

- **A fixed set of destinations** — the routes, each with a clear description (here: billing, technical, account, general).
- **A decision engine** — rules, a classifier, embeddings, or an LLM — that maps a request to one route.
- **A constrained output** — for the LLM routers, a schema so the model can only return a valid route label (this lab uses pydantic).
- **A confidence signal (optional)** — a score the router can defer on, sending low-confidence cases to a human.
- **A test set + a metric** — labeled examples so you can measure accuracy and compare engines.

## Considerations for trustworthy routing

- **Wrong routes are silent failures.** A misrouted request often looks handled but isn't — measure accuracy on real data, don't assume.
- **Constrain the output.** Free-text labels drift ("Billing", "billing dept", "money"). A schema (pydantic `Literal`) forces exactly one valid route.
- **Gate on confidence.** When the router is unsure, escalating to a human is safer than a confident wrong guess.
- **Match cost to value.** The cheapest engine that meets your accuracy bar usually wins; reserve the LLM router for genuinely ambiguous input.
- **Watch for drift.** New kinds of request appear over time; revisit the routes and the test set periodically.

## A primer: list comprehensions, generators, and `**kwargs`

The routers below use three Python features that newer programmers sometimes find tricky. Each has a tiny runnable example — **run the cell, then tweak it** to see what changes.

### 1. List comprehension — build a new list from a loop

Square brackets `[ ]` run a loop and collect every result into a **new list**. The lab uses this kind of one-line list building for route names and labels.

In [ ]:
# A list comprehension builds a NEW list by looping over an existing one.
routes = ["billing", "technical", "account", "general"]

shouting = [name.upper() for name in routes]   # transform each item
print(shouting)

### 2. Generator expression with `any()` — match against a list

A **generator expression** looks like a list comprehension but uses `( )` instead of `[ ]`, and it produces **one item at a time**. Handed to `any()`, it checks the items and stops at the first match. The **rule-based router** uses exactly this to test a message against a route's keywords.

In [ ]:
# Generator expression inside any(): scans the keywords and stops at the
# first one found in the message. (This is the rule-based router's match check.)
keywords = ["charge", "refund", "invoice"]
message = "i want a refund please".lower()

print(any(word in message for word in keywords))   # -> True

### 3. Generator expression with `.join()` — build a string

`.join()` takes the items a generator produces and glues them into **one string**, putting the separator *between* each item. The **LLM routers** use this to build the "menu" of routes they show the model. The separator can be anything — run each block below to see a different one.

In [ ]:
# .join(SEP) glues a sequence of STRINGS into one string, with SEP placed
# *between* the items (never at the ends).
routes = {"billing": "charges and refunds", "technical": "bugs and errors"}

# Newline separator -> a multi-line "menu" (exactly what the routers build).
menu = "\n".join(f"- {name}: {desc}" for name, desc in routes.items())
print(menu)

In [ ]:
# Comma separator -> the route names on one inline line.
# (Looping a dict yields its keys, so this joins "billing" and "technical".)
print(", ".join(routes))

In [ ]:
# Any separator works -- here a " | " divider.
print(" | ".join(routes))

In [ ]:
# join() only works on strings, so convert numbers with str() first.
counts = [3, 1, 4]
print("-".join(str(n) for n in counts))   # -> "3-1-4"

### 4. `**kwargs` — collect keyword arguments into a dict

`**kwargs` lets a function accept **any keyword arguments** and bundle them into a dict, then forward them on. The lab's `_call` helper uses it so one small function can serve both `generate()` (which passes `contents`) and `generate_json()` (which also passes `config`).

In [ ]:
# ** UNPACKS a dict back into keyword arguments: greet(**person) is the same as
# passing each key as its own argument by hand.
def greet(name, job, satisfaction):
    print(f"{name} works as a {job} and is feeling {satisfaction} about their job right now.")

person = {"name": "Ana", "job": "data scientist", "satisfaction": "very happy"}
greet(**person)     # same as greet(name="Ana", job="data scientist", satisfaction="very happy")

person2 = {"name": "Marco", "job": "support engineer", "satisfaction": "a bit frustrated"}
greet(**person2)

> **What this lab spends:** the routers run over an 8-message test set. The two LLM routers make ~1 call per message, so a full run is roughly **30 requests**.

## Setup: add your Gemini API key as a Colab secret

1. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey).
2. In Colab, click the **key icon** in the left sidebar ("Secrets").
3. Add a new secret named **`GEMINI_API_KEY`** and paste your key as the value.
4. Toggle **"Notebook access"** on for that secret.

The next cell installs the SDK and reads your key.

In [ ]:
!pip install -q google-genai python-dotenv

In [ ]:
import time
from typing import Literal

from google.colab import userdata
from dotenv import load_dotenv
from google import genai
from google.genai import errors
from pydantic import BaseModel, Field

load_dotenv()

api_key = userdata.get('GEMINI_API_KEY')
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (key icon) and enable notebook access.")

client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash-lite"

# Quick connectivity check
print(client.models.generate_content(model=MODEL, contents="Say 'Setup complete!' and nothing else.").text)

## Helpers: text and structured JSON

The router strategies below need the model in two shapes. Both route through `_call`, which waits and retries once on a rate limit (HTTP 429).

- `generate()` — prose back.
- `generate_json(prompt, schema)` — schema-validated JSON; `response.parsed` returns typed Python objects (a `pydantic` model or a typed list). No `json.loads`, no regex.

In [ ]:
# Three small helpers wrap the Gemini SDK so the rest of the lab stays readable.

# _call is the low-level call the two public helpers below share. `**kwargs` means
# "collect any keyword arguments into a dict and pass them straight through to Gemini" --
# so generate() can hand it `contents=...` while generate_json() also hands it `config=...`,
# all through this one function. It also retries on the two transient errors you are most
# likely to hit when running many calls back to back (see the head-to-head section):
#   * 429 (rate limited)      -> wait 60s, then try again
#   * 503 (model overloaded)  -> wait 10s, then try again
def _call(**kwargs):
    """Call Gemini, retrying on transient 429 (rate limit) and 503 (overloaded) errors."""
    for attempt in range(4):  # the first call + up to 3 retries
        try:
            return client.models.generate_content(model=MODEL, **kwargs)
        except errors.ClientError as e:
            # 429 = too many requests (rate limit).
            if e.code == 429 and attempt < 3:
                print("  Rate limited (429). Waiting 60s, then retrying...")
                time.sleep(60)
            else:
                raise
        except errors.ServerError as e:
            # 503 = model temporarily overloaded ("high demand"). A short wait usually clears it.
            if e.code == 503 and attempt < 3:
                print("  Model overloaded (503). Waiting 10s, then retrying...")
                time.sleep(10)
            else:
                raise


# generate: use this when you want plain TEXT back (a sentence, a paragraph).
def generate(prompt: str) -> str:
    """Send a single prompt to Gemini and return the response text."""
    return (_call(contents=prompt).text or "").strip()


# generate_json: use this when you want STRUCTURED data back instead of prose. We hand
# Gemini a `schema` (a pydantic model or typed list) and turn on JSON mode, so
# `response.parsed` comes back already validated into that shape -- no manual parsing.
def generate_json(prompt: str, schema):
    """Generate schema-validated JSON. `schema` is a pydantic model or typed list."""
    response = _call(
        contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": schema},
    )
    return response.parsed

## The shared task: a customer-support router

Every router below answers the same question: *which queue does this incoming message belong to?* The destinations and a labeled test set are fixed up front so we can compare routers fairly.

In [ ]:
# The fixed set of destinations. The key is the route name; the value is a plain-English
# description that the LLM-based routers show the model as the menu of choices.
ROUTES = {
    "billing":   "Invoices, charges, refunds, payment methods, pricing, or subscription costs.",
    "technical": "Bugs, errors, crashes, login failures, things not working, or how-to technical help.",
    "account":   "Account access, password resets, profile changes, cancellations, or plan changes.",
    "general":   "Greetings, business questions, feedback, or anything that fits no other category.",
}
# list(a_dict) returns its keys, so CATEGORIES == ["billing", "technical", "account", "general"].
CATEGORIES = list(ROUTES)

# Labeled test set: (message, correct_route). We'll score every router against this.
TEST_MESSAGES = [
    ("I was charged twice for my subscription this month.",        "billing"),
    ("Your software keeps freezing whenever I export a report.",   "technical"),
    ("How do I reset my password? The link never arrives.",        "account"),
    ("What are your customer service hours?",                      "general"),
    ("I'd like a refund for the annual plan I bought yesterday.",  "billing"),
    ("The login page just spins forever and never loads.",         "technical"),
    ("Please cancel my account and delete my data.",               "account"),
    ("Thanks so much for the help last week!",                     "general"),
]

print(f"{len(CATEGORIES)} routes, {len(TEST_MESSAGES)} labeled test messages.")

---

## Strategy 1 — Rule-Based Routing

The simplest router: **explicit logic** — keywords, regexes, metadata, fixed conditions. No model call, so it is instant, free, fully transparent, and 100% reproducible. This is the right tool when your workflows are predictable and you need tight control.

```
   message.lower()
        │
        ▼
   ┌──────────────────────────────┐   match
   │ contains "charge" / "refund"? │ ───────▶ [ billing ]
   └──────────────────────────────┘
        │ no
        ▼
   ┌──────────────────────────────┐   match
   │ contains "crash" / "error"?   │ ───────▶ [ technical ]
   └──────────────────────────────┘
        │ no
        ▼
   ┌──────────────────────────────┐   match
   │ contains "password"/"cancel"? │ ───────▶ [ account ]
   └──────────────────────────────┘
        │ no
        ▼
   [ general ]   ← default: nothing matched
```

In [ ]:
# Each rule is a (route, keywords) pair, checked top to bottom -- so put the most specific
# routes first. There is no model call here: this router is pure, instant Python.
RULES = [
    ("billing",   ["charge", "charged", "refund", "invoice", "payment", "price", "bill", "subscription cost"]),
    ("technical", ["crash", "freez", "error", "bug", "broken", "not working", "won't load", "spins"]),
    ("account",   ["password", "cancel", "delete my", "profile", "upgrade", "downgrade", "log in", "login"]),
]


# route_rules lowercases the message and returns the first route whose keyword list
# matches; if none match it falls back to "general".
def route_rules(message: str) -> str:
    """Match the message against keyword lists, falling back to 'general'."""
    text = message.lower()
    for route, keywords in RULES:
        # any(kw in text for kw in keywords) is a generator expression: it scans the
        # keywords one by one and is True the moment one of them appears in the message.
        if any(kw in text for kw in keywords):
            return route
    return "general"


for msg, _ in TEST_MESSAGES:
    print(f"{route_rules(msg):10s} <- {msg}")

**The catch:** rules only know the words you gave them. A user who writes *"you billed me for something I never bought"* slips through — there's no `bill` keyword match for `billed`... actually there is, but *"my last payment didn't go through and now I have no access"* is genuinely ambiguous, and *"this thing is completely useless"* (a technical complaint) matches nothing and falls to `general`. Rules are brittle exactly where language is flexible.

In [ ]:
tricky = [
    "This thing is completely useless and I'm furious.",   # technical complaint, no keyword
    "My card got declined so now I can't get in.",         # billing? account? rules guess
]
for msg in tricky:
    print(f"{route_rules(msg):10s} <- {msg}")

---

## Strategy 2 — Classifier-Based Routing

Hand the decision to an LLM, but **constrain it to your predefined categories** with a schema. `Literal[...]` means the model *cannot* return anything but one of the four valid labels — no prose, no invented category, no parsing. This is the workhorse when the destinations are known in advance and you want the model's language understanding without giving it free rein.

```
   "I was charged twice"  ─────┐
   (message)                   │
                               ▼
                       ┌────────────────────────┐        ┌─────────────┐
   route descriptions  │          LLM           │        │  billing    │ ◀── only one
   billing  : ...   ──▶│  pick exactly ONE      │ ─────▶ │  technical  │     of these
   technical: ...      │  schema = Literal[...] │        │  account    │     four can
   account  : ...      └────────────────────────┘        │  general    │     come back
   general  : ...                  ▲
                                   └── the schema makes any other output impossible
```

In [ ]:
# Classification is a pydantic model -- a typed schema for the LLM's answer. `Literal[...]`
# pins `category` to exactly these four strings, so the model cannot return anything else
# and pydantic validates the reply for us.
class Classification(BaseModel):
    category: Literal["billing", "technical", "account", "general"]


# route_classifier asks the LLM to pick one category, constrained by the schema above.
def route_classifier(message: str) -> str:
    """Classify the message into one of the predefined routes via the LLM."""
    # The part inside join() is a generator expression: it produces one "- name: description"
    # line per (name, description) pair in ROUTES -- the menu we show the model.
    routes_desc = "\n".join(f"- {name}: {desc}" for name, desc in ROUTES.items())
    result = generate_json(
        "Classify the customer-support message into exactly one category.\n\n"
        f"Categories:\n{routes_desc}\n\n"
        f"MESSAGE: {message}",
        schema=Classification,
    )
    return result.category


for msg, _ in TEST_MESSAGES[:4]:
    print(f"{route_classifier(msg):10s} <- {msg}")

---

## Strategy 3 — Embedding / Semantic Routing (preview)

The third strategy routes by **meaning** rather than keywords or reasoning: embed a few example utterances per route once, embed the incoming message, then send it to the route whose examples sit closest in vector space (highest cosine similarity). It's cheap — one embedding, no LLM reasoning — and robust to wildly varied phrasing: *"freezes"*, *"locks up"*, and *"hangs"* all land in the same neighborhood.

> **We'll build this in Week 8 (Retrieval-Augmented Generation).** Semantic routing runs on the exact machinery RAG is built from — text embeddings plus cosine similarity — so we introduce it there. Once you've done the RAG lab you'll have everything needed to drop an embedding router into the comparison below (see the final exercise). For now we implement and compare the other three strategies.

---

## Strategy 4 — LLM-as-Router (with a confidence gate)

The most flexible router: let the LLM **reason** about the request and return not just a route but its **confidence** and a short rationale. That confidence is the key that unlocks **confidence-gated routing** — when the model isn't sure enough, we don't force a guess; we **escalate to a human**. This is the right choice for ambiguous or context-dependent decisions where a wrong auto-route is worse than a brief human review.

```
   message
      │
      ▼
   ┌───────────────────────────┐
   │  LLM reasons about intent │
   └───────────────────────────┘
      │
      ▼
   { category, confidence, reasoning }
      │
      ▼
   ╔═══════════════════════════╗
   ║   confidence ≥ 0.6 ?      ║
   ╚═══════════════════════════╝
      │                     │
   yes│                     │no
      ▼                     ▼
  ┌──────────────────┐   ┌────────────────────────┐
  │ route to category│   │ escalate → human review│
  └──────────────────┘   │ (don't force a guess)  │
                         └────────────────────────┘
```

In [ ]:
# RouterDecision is a pydantic model with THREE fields, so the LLM returns not just a
# category but also a confidence and a reasoning. Field(description=...) documents a field;
# that text is sent to the model so it knows what to put there. pydantic validates the reply.
class RouterDecision(BaseModel):
    category: Literal["billing", "technical", "account", "general"]
    confidence: float = Field(description="How confident, from 0.0 to 1.0")
    reasoning: str = Field(description="One sentence explaining the choice")


# route_llm lets the model reason and return the structured decision above.
def route_llm(message: str) -> RouterDecision:
    """Ask the LLM to reason about the request and return a structured decision."""
    # Same generator-expression menu as the classifier: one "- name: description" line per route.
    routes_desc = "\n".join(f"- {name}: {desc}" for name, desc in ROUTES.items())
    return generate_json(
        "You are a support-desk router. Choose the best category for the message, "
        "rate your confidence honestly (low if the message is vague or spans categories), "
        "and explain in one sentence.\n\n"
        f"Categories:\n{routes_desc}\n\n"
        f"MESSAGE: {message}",
        schema=RouterDecision,
    )


d = route_llm("I was charged twice for my subscription this month.")
print(f"category={d.category}  confidence={d.confidence}  reasoning={d.reasoning}")

In [ ]:
CONFIDENCE_THRESHOLD = 0.6


# route_with_gate runs the LLM router, then applies a confidence gate: if the model is sure
# enough we trust its category; otherwise we send the message to a human ("escalate").
def route_with_gate(message: str):
    """Route via the LLM, but escalate to a human when confidence is too low."""
    d = route_llm(message)
    # A conditional expression (value_if_true if condition else value_if_false): keep the
    # model's category when confidence clears the threshold, otherwise "escalate".
    destination = d.category if d.confidence >= CONFIDENCE_THRESHOLD else "escalate"
    return destination, d


# A clear case routes automatically; a vague/charged one gets escalated for human review.
for msg in [
    "The export button throws a 500 error every time.",
    "Honestly nothing about this works and I'm done, someone needs to fix this NOW.",
]:
    dest, d = route_with_gate(msg)
    flag = "→ HUMAN REVIEW" if dest == "escalate" else ""
    print(f"[{dest:9s}] conf={d.confidence:.2f} {flag}\n   {msg}\n   reason: {d.reasoning}\n")

---

## Head-to-head: which router is most accurate?

Now run all four over the labeled test set and score them. The point isn't that one router always wins — it's that **each makes a different trade-off** between cost, speed, transparency, and flexibility. Measure on *your* data before you choose.

In [ ]:
# A dict mapping each router's name to the function that runs it. The "llm" entry uses a
# lambda -- a one-line throwaway function -- to call route_llm and keep only its .category,
# so all three routers share the same "message in, label out" shape.
routers = {
    "rule-based": route_rules,
    "classifier": route_classifier,
    "llm":        lambda m: route_llm(m).category,
}

# Per-message predictions
# Header row. The join holds a generator expression that prints each router name in an
# 11-character-wide column.
header = f"{'message':42s} {'gold':10s} " + " ".join(f"{name:11s}" for name in routers)
print(header)
print("-" * len(header))
# scores is a dict comprehension that starts every router at 0; we add 1 per correct answer.
scores = {name: 0 for name in routers}
for msg, gold in TEST_MESSAGES:
    # preds is a dict comprehension: run every router on this message -> {name: predicted_route}.
    preds = {name: fn(msg) for name, fn in routers.items()}
    for name, pred in preds.items():
        if pred == gold:
            scores[name] += 1
    # Build the row: each prediction plus a check or cross, joined into fixed-width columns.
    cells = " ".join(f"{(p + ('✓' if p == gold else '✗')):11s}" for p in (preds[n] for n in routers))
    print(f"{msg[:40]:42s} {gold:10s} {cells}")

print("\nAccuracy:")
n = len(TEST_MESSAGES)
for name, s in scores.items():
    print(f"  {name:11s} {s}/{n}  ({s / n:.0%})")

## How to choose

| If you need… | Reach for |
| :--- | :--- |
| Speed, zero cost, total transparency, predictable inputs | **Rule-based** |
| Solid accuracy over a *fixed* set of categories | **Classifier-based** |
| Robustness to wildly varied phrasing, cheap per-query | **Embedding / semantic** *(built in Week 8)* |
| Reasoning over ambiguity + a confidence signal to gate on | **LLM-as-router** |

In production these compose: a common pattern is **rules first** (catch the obvious, free cases), **fall back to an LLM router** for everything else, and **escalate to a human** when even the LLM's confidence is low. You get the speed of rules, the flexibility of the LLM, and a safety net — best of all four.

## Your turn (exercises)

1. **Multi-label routing.** Some messages belong to two queues ("I was charged after I cancelled" = billing **and** account). Change the classifier's schema to `list[Literal[...]]` and return every applicable route.
2. **Tune the gate.** Sweep `CONFIDENCE_THRESHOLD` from 0.4 to 0.9 on the test set. How does the escalation rate trade off against accuracy on the messages that *aren't* escalated?
3. **Grow a route.** Add a fifth category (e.g. `sales`). Notice the cost of each router: rules need new keywords, embedding needs new examples, the LLM routers need one line in the description. Which scaled most cheaply?
4. **Hybrid router.** Write `route_hybrid(message)` that tries `route_rules` first and only calls `route_llm` when the rules fall through to `general`. Compare its accuracy *and* its LLM-call count to the pure LLM router.


When you're done, save a copy (**File → Save a copy in Drive**) and submit your notebook link via Canvas.